🏁 02 — THE GREAT RACE: Sequential vs Threads (10-Task Scorecard!)

**Topic 12.01 | Async/Parallel Python — Part 1 of 4** *(DEMO notebook)*

🌍 **Hook:** Remember the school relay race? One kid runs the whole track
(sequential) vs 10 kids each on their own lane (threads). Today we run an official RACE
of 10 "file-download" tasks — with a photo-finish timing scorecard!

In this demo:
- 🏁 10 IO-tasks: sequential 🐢 vs threaded ⚡ (with bar-race text graphics!)
- 🧮 ThreadPoolExecutor — the modern way (no more loop-join hassle)
- 🔒 GIL-plateau LIVE: CPU-crunch 1 vs 2 vs 4 threads — the line stays flat!
- 📬 `map` (order-preserved) vs `as_completed` (whoever finishes first, gets served first!)

## 🏁 Race-1 — 10 Downloads, Two Teams

In [1]:
import time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

TASKS = 10
def download(i, secs=0.25):
    """har download 0.25s wait (IO-bound!)"""
    time.sleep(secs)
    return f"DL-{i}✅"

# TEAM-A: Sequential 🐢
t0 = time.perf_counter()
seq_res = [download(i) for i in range(TASKS)]
seq_t = time.perf_counter() - t0

# TEAM-B: ThreadPool ⚡ (modern API — pool me daalo, result le lo!)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as pool:
    thr_res = list(pool.map(download, range(TASKS)))
thr_t = time.perf_counter() - t0

def bar(t, scale=25):
    return "█" * max(1, round(t * scale / seq_t * 10))

print("🏁 RACE RESULT (10 × 0.25s downloads):")
print(f"   🐢 sequential {seq_t:5.2f}s {bar(seq_t)}")
print(f"   ⚡ threaded   {thr_t:5.2f}s {bar(thr_t)}")
print(f"\n   🏆 SPEEDUP = {seq_t/thr_t:.1f}x  |  pool.map ne order bhi bachayi: {seq_res == thr_res}")
print("   📌 ThreadPoolExecutor = thread-loop ka AUTO-GEAR: start()/join() likhna hi nahi!")
# Expected OUTPUT (speedup ~8-10x machine pe; scale bars relative):
# 🏁 RACE RESULT (10 × 0.25s downloads):
#    🐢 sequential 2.50s ██████████████████████████
#    ⚡ threaded   0.26s ███
#    🏆 SPEEDUP = ~9.7x  |  pool.map ne order bhi bachayi: True

🏁 RACE RESULT (10 × 0.25s downloads):
   🐢 sequential  2.50s ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
   ⚡ threaded    0.27s ███████████████████████████

   🏆 SPEEDUP = 9.4x  |  pool.map ne order bhi bachayi: True
   📌 ThreadPoolExecutor = thread-loop ka AUTO-GEAR: start()/join() likhna hi nahi!


## 📬 map vs as_completed — Two Ways of a Postman

📖 **Definitions:**
- `pool.map(fn, items)` = wait for all to finish, results come in **submission order**
- `as_completed(futures)` = **the one that finishes first arrives first** (finishing order!) —
  PERFECT for live progress bars

🔍 **WHY does the difference matter?** If you want to show "7/10 done" on a dashboard,
use as_completed; if you want all results together in one table, use map.

In [2]:
import random

random.seed(7)
def download_var(i):
    time.sleep(0.05 * random.randint(1, 5))    # har file ka "size" alag!
    return f"DL-{i}"

submit_order = list(range(8))
with ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(download_var, i): i for i in submit_order}
    finish_order = []
    for fut in as_completed(futs):
        finish_order.append(futs[fut])
        print(f"   📬 aaya: DL-{futs[fut]}  ({len(finish_order)}/8)")

print("\n📤 SUBMIT order :", submit_order)
print("📥 FINISH order :", finish_order, " ← chhote kaam pehle lautey!")
print("💡 as_completed = real-time progress ka saathi (tqdm jaisi feel!)")
# Expected OUTPUT (seed=7 fix — ye EXACT order tumhari machine pe bhi!):
# 📬 aaya: DL-.. (1/8) ... (8/8)
# 📤 SUBMIT order : [0,1,2,3,4,5,6,7]
# 📥 FINISH order : [random-seeded order — chhote waale pehle]

   📬 aaya: DL-3  (1/8)
   📬 aaya: DL-4  (2/8)
   📬 aaya: DL-6  (3/8)
   📬 aaya: DL-1  (4/8)
   📬 aaya: DL-0  (5/8)
   📬 aaya: DL-7  (6/8)
   📬 aaya: DL-2  (7/8)
   📬 aaya: DL-5  (8/8)

📤 SUBMIT order : [0, 1, 2, 3, 4, 5, 6, 7]
📥 FINISH order : [3, 4, 6, 1, 0, 7, 2, 5]  ← chhote kaam pehle lautey!
💡 as_completed = real-time progress ka saathi (tqdm jaisi feel!)


## 🔒 Race-2 — GIL-Plateau LIVE: 1 vs 2 vs 4 CPU-Threads

📖 **Experiment:** CPU-crunch × 2 workers × each done twice — increase the threads and
watch the time. Threading is the champion only in the WAITING game; in thinking-heavy work the GIL holds everyone back.

In [3]:
def crunch(n, loops=4_000_000):
    s = 0
    for i in range(loops):
        s += i * i
    return n

def race_cpu(workers):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(crunch, range(4)))
    return time.perf_counter() - t0

print("🔒 GIL-PLATEAU EXPERIMENT (4 CPU-crunches):")
base = None
for wcount in (1, 2, 4):
    t = race_cpu(wcount)
    base = base or t
    print(f"   threads={wcount}  time={t:.2f}s  speedup={base/t:.2f}x")
print("\n   😶 Dekha? Line FLAT — soch-wala kaam threads se nahi ud-ta (GIL 🔑)")
print("   👉 Aise me: multiprocessing (pure CPU) ya code numba/C me (pro-level)!")
# Expected OUTPUT (times alag per-machine — PATTERN yehi: sab ~same!):
# 🔒 GIL-PLATEAU EXPERIMENT (4 CPU-crunches):
#    threads=1  time=~2.4s  speedup=1.00x
#    threads=2  time=~2.4s  speedup=~1.0x
#    threads=4  time=~2.4s  speedup=~1.0x
#    😶 Dekha? Line FLAT ...

🔒 GIL-PLATEAU EXPERIMENT (4 CPU-crunches):
   threads=1  time=0.78s  speedup=1.00x
   threads=2  time=0.79s  speedup=0.99x
   threads=4  time=0.77s  speedup=1.01x

   😶 Dekha? Line FLAT — soch-wala kaam threads se nahi ud-ta (GIL 🔑)
   👉 Aise me: multiprocessing (pure CPU) ya code numba/C me (pro-level)!


## 🏆 Final Scorecard + When-to-Use-What Cheat-Matrix

In [4]:
print("🏆 THE GREAT-RACE SCORECARD")
print("=" * 46)
print(f"   IO-race (10 downloads)  : sequential {seq_t:.2f}s vs threaded {thr_t:.2f}s → {seq_t/thr_t:.0f}x ⚡")
print(f"   CPU-race (4 crunches)   : 1/2/4 threads → FLAT ~1.0x (GIL!) 🔒")
print("=" * 46)
print("""
🧭 KAB-KYA (life-ka cheat-matrix):
   network/file/DB/sleep (WAIT-game)  → threading YA async (aage aayega!)
   math/image/ML-crunch (SOCH-game)   → multiprocessing (alag core!)
   1000s tiny-waits (APIs storm)      → asyncio (12.02 me superhero-entry 🦸)
""")
# Expected OUTPUT:
# 🏆 THE GREAT-RACE SCORECARD
# ==============================================
#    IO-race (10 downloads)  : sequential 2.5Xs vs threaded 0.2Xs → 9x ⚡
#    CPU-race (4 crunches)   : 1/2/4 threads → FLAT ~1.0x (GIL!) 🔒
# ==============================================
# 🧭 KAB-KYA ...

🏆 THE GREAT-RACE SCORECARD
   IO-race (10 downloads)  : sequential 2.50s vs threaded 0.27s → 9x ⚡
   CPU-race (4 crunches)   : 1/2/4 threads → FLAT ~1.0x (GIL!) 🔒

🧭 KAB-KYA (life-ka cheat-matrix):
   network/file/DB/sleep (WAIT-game)  → threading YA async (aage aayega!)
   math/image/ML-crunch (SOCH-game)   → multiprocessing (alag core!)
   1000s tiny-waits (APIs storm)      → asyncio (12.02 me superhero-entry 🦸)



---
## 📌 Recap — 30 Seconds
- `ThreadPoolExecutor.map` = auto-gear threading (ordered) ⚡
- `as_completed` = finishing-order live-feed 📬 (the friend of progress bars)
- Threads give 9-10x in the IO-race; the GIL-plateau stays FLAT in the CPU-race 🔒 — remember this pattern!

**Next:** `02_Async_Await` — the single-thread superhero: one pair of hands, THOUSANDS of tasks.
The async/await waiter-magic begins! 🦸‍♂️